In [ ]:
# -*- coding: utf-8 -*-
"""
fNIRS processing (epoch-based) with configurable pipeline switches:
OD -> [LinearDetrend] -> [SCI(drop)] -> [TDDR] -> [SCR] -> [BPF]
-> [PostFilter Channel QC (range/var drop)] -> [BAD power annotations]
-> [keep long] -> [prune/order wavelength pairs] -> MBLL(Hb)

Global baseline: use the MIDDLE X minutes of the single 'Baseline' at experiment start.

Final visualization:
- For chosen subjects & activities, one window per (Subject, Activity).
- Window shows 4 subplots: NG-T1, G-T1, NG-T2, G-T2.
- In each subplot, plot ALL HbO channel time series (no averaging). Missing trials are labeled.

NEW (ROI path):
- Compute ROI-averaged waveforms (after all cleaning/masking/baseline).
- Compute features on ROI-averaged waveforms; save N_ch used per ROI.
- Save ROI-averaged time series to Excel (wide).
- Save ROI overlay plots (10 ROIs per subplot) to roi_plots/ with legends.

NEW (Both ROI methods):
- Method 1 (ChWise_*): features per channel, then averaged across ROI channels.
- Method 2 (ROIWise_*): average waveform per ROI, then features from that waveform.
- Save a combined Excel comparing both: roi_features_both_methods.xlsx

QC outputs per subject (in OUT_QC_DIR):
- <Subject>_sci.csv
- <Subject>_chanvar_qc.csv
- <Subject>_baseline.csv
- <Subject>_trial_qc.csv
- <Subject>_channel_inclusion_by_trial.csv
"""

import re
from pathlib import Path
import numpy as np
import pandas as pd
import mne
import mne_nirs
import matplotlib.pyplot as plt

# ---------------- CONFIG ----------------
CF_DIR = r"...\annotated_fif_files_CF"
SD_DIR = r"...\annotated_fif_files_SD"
TEMPLATE_XLSX = "all_subjects_expected_trials.xlsx"

# Existing per-channel outputs (unchanged)
OUT_SUMMARY = "features_get_per_channel_then_average.xlsx"
OUT_TIMESER = "timeseries_get_per_channel_then_average.xlsx"

# NEW: ROI outputs
OUT_ROI_SUMMARY = "features_channelAverageFirst.xlsx"      # includes N_ch
OUT_ROI_TIMESER = "timeseries_channelAverageFirst.xlsx"
OUT_ROI_FIG_DIR = "perROI_HbO"  # images with 2x2 panels and ~10 ROI lines per panel

# NEW: Combined ROI methods comparison
OUT_ROI_METHODS_SUMMARY = "roi_features_both_methods.xlsx"

OUT_QC_DIR  = "qc_logs"  # per-subject CSVs

# Which subjects to include (others ignored even if in template)
SUBJECTS_CF = ["S1"]      # subset of S1..S24
SUBJECTS_SD = ["S26"]     # subset of S25..S48
#SUBJECTS_CF = [f"S{i}" for i in range(1, 25)]
#SUBJECTS_SD = [f"S{i}" for i in range(25, 49)]

# Trial ordering controls
ACTIVITIES   = ['TWEO','TWEC','TWDT','OW','HOW','FPEONF','FPECNF','FPEODT','FPECDT','FPEOWF','FPECWF','FNC','PP']
STIMS_ORDER  = ['NG', 'G']
TRIALS_ORDER = ['T1', 'T2']

# ---------- Pipeline Toggles ----------
DO_OD_LINEAR_DETREND = False
DO_SCI               = True
DO_TDDR              = True
DO_SCR               = True
DO_BPF               = True
DO_POSTFILTER_VAR_QC = True
DO_POWER_ANNOT       = False
KEEP_LONG_ONLY       = True
DO_BASELINE_SUBTRACT = True
EXPORT_TIMESER       = False
DO_PLOTS             = False

# ---------- Visualization Toggles ----------
#PLOT_SUBJECTS   = ["S1"]
PLOT_SUBJECTS = SUBJECTS_CF + SUBJECTS_SD
#PLOT_ACTIVITIES = ["TWEO"]
PLOT_ACTIVITIES = ACTIVITIES
PLOT_SMOOTH_SEC = 0.0
PLOT_TIME_UNITS = "s"        # "s" or "samples"
PLOT_SAVE_FIGS  = False
PLOT_SHOW       = False
OUT_FIG_DIR     = "Per_Channel_HbO"

# Processing tunables
SCI_THRESHOLD = 0.60
LONG_DIST_CM  = 1.0
BPF_LF, BPF_HF = 0.01, 0.5
IIR_ORDER = 5

# Post-filter QC
QC_METRIC = "range"           # "range" or "var"
QC_MULT   = 5.0
QC_MIN_RANGE = 1e-10

# Power-based annotator
POWER_WIN_S  = 6.0
POWER_STEP_F = 0.25
POWER_Z_TH   = 3.0
POWER_MASK_DROP_FRAC = 0.80

# Global baseline: use MIDDLE X minutes of the single baseline annotation
BASELINE_MIDDLE_MIN = 3.0

# ---- Region definitions (HbO channels only) ----
# Channel names must match HbO channel labels in hb.ch_names (e.g., "S1_D1 hbo")
brain_regions = {
    "OFC": ["S1_D1 hbo", "S1_D3 hbo", "S2_D3 hbo", "S16_D3 hbo", "S2_D2 hbo"],
    "L_DLPFC": ["S4_D1 hbo", "S4_D4 hbo", "S16_D1 hbo", "S16_D4 hbo"],
    "R_DLPFC": ["S5_D2 hbo", "S5_D5 hbo", "S16_D2 hbo", "S16_D5 hbo"],
    "SMA": ["S3_D5 hbo", "S3_D4 hbo", "S3_D6 hbo", "S6_D4 hbo", "S6_D6 hbo", "S6_D9 hbo",
            "S7_D5 hbo", "S7_D6 hbo", "S7_D10 hbo", "S9_D6 hbo"],
    "L_PMC": ["S4_D7 hbo", "S6_D7 hbo", "S8_D7 hbo"],
    "R_PMC": ["S5_D8 hbo", "S7_D8 hbo", "S10_D8 hbo"],
    "PMC": ["S8_D9 hbo", "S9_D9 hbo", "S9_D10 hbo", "S10_D10 hbo"],
    "LSG": ["S11_D11 hbo", "S8_D11 hbo"],
    "RSG": ["S12_D12 hbo", "S10_D12 hbo"],
    "PVC": ["S13_D13 hbo", "S13_D14 hbo", "S13_D15 hbo", "S14_D15 hbo",
            "S14_D13 hbo", "S15_D14 hbo", "S15_D15 hbo"],
}

# ------------- HELPERS ------------------

def fif_path_for_subject(subj: str) -> Path:
    n = int(subj[1:])
    if n <= 24:
        return Path(CF_DIR) / f"{subj}-pr-annotated_raw.fif"
    src = f"S{n-24}"
    return Path(SD_DIR) / f"{src}-6pm-annotated_raw.fif"

def pick_fnirs_channels(raw: mne.io.BaseRaw) -> np.ndarray:
    picks = mne.pick_types(raw.info, fnirs=True, stim=False, eeg=False, meg=False)
    if picks.size == 0:
        picks = mne.pick_types(raw.info, misc=True, stim=False, eeg=False, meg=False)
        if picks.size == 0:
            stim = mne.pick_types(raw.info, stim=True)
            picks = np.setdiff1d(np.arange(raw.info['nchan']), stim)
    return picks

def _pair_label(ch_name: str) -> str:
    m = re.match(r'^(S\d+_D\d+)\s', str(ch_name))
    return m.group(1) if m else ch_name

def is_bad_overlap(raw: mne.io.BaseRaw, s_samp: int, e_samp: int) -> bool:
    sf = raw.info['sfreq']
    t0 = raw.first_time + s_samp / sf
    t1 = raw.first_time + e_samp / sf
    for on, dur, desc in zip(raw.annotations.onset, raw.annotations.duration, raw.annotations.description):
        d = str(desc).upper()
        if "BAD" in d and not d.startswith("BAD_POWER"):
            if not (on + dur <= t0 or on >= t1):
                return True
    return False

def annotate_high_power_segments(raw_od: mne.io.BaseRaw,
                                 win_sec=POWER_WIN_S, step_frac=POWER_STEP_F, z_th=POWER_Z_TH):
    from scipy.signal import welch
    sf = raw_od.info['sfreq']
    data = raw_od.get_data()
    win  = max(1, int(win_sec * sf))
    step = max(1, int(win * step_frac))
    bad_spans = {}
    for ci, ch in enumerate(raw_od.ch_names):
        if win >= data.shape[1]:
            continue
        x = data[ci]
        powers, starts = [], []
        for start in range(0, data.shape[1] - win, step):
            seg = x[start:start+win]
            _, Pxx = welch(seg, fs=sf, nperseg=min(win, 2048))
            powers.append(Pxx.sum()); starts.append(start)
        powers = np.asarray(powers)
        if powers.size < 10 or not np.isfinite(powers).any():
            continue
        z = (powers - np.nanmean(powers)) / (np.nanstd(powers) + 1e-12)
        for start, zi in zip(starts, z):
            if zi > z_th:
                bad_spans.setdefault(ch, []).append((start, start + win))
                onset = raw_od.first_time + start / sf
                raw_od.annotations.append(onset=float(onset), duration=float(win_sec),
                                          description=f'BAD_power_{ch}')
    return bad_spans

def separate_long_short(raw_or_info, long_thresh_m=LONG_DIST_CM/100.0):
    info = raw_or_info.info if hasattr(raw_or_info, "info") else raw_or_info
    dists = mne.preprocessing.nirs.source_detector_distances(info)
    long_names, short_names = [], []
    for ch_name, dist in zip(info.ch_names, dists):
        if dist is None:
            continue
        if dist >= long_thresh_m:
            long_names.append(ch_name)
        else:
            short_names.append(ch_name)
    return long_names, short_names

def parse_trials(hb: mne.io.BaseRaw):
    sf = float(hb.info['sfreq'])
    spans = {}
    for on, dur, desc in zip(hb.annotations.onset, hb.annotations.duration, hb.annotations.description):
        d = str(desc).strip()
        if d.lower() == "baseline":
            continue
        m = re.match(r'^.+-([A-Za-z]+)-([A-Za-z]+)-(T[12])$', d)
        if not m:
            continue
        act, stim, tr = m.groups()
        s = int((on - hb.first_time) * sf)
        e = int((on - hb.first_time + dur) * sf)
        s = max(0, s); e = min(hb.n_times, e)
        if e > s:
            key = (act.upper(), stim.upper(), tr.upper())
            spans[key] = (s, e, d)
    return spans


def compute_epoch_features(seg: np.ndarray, sf: float):
    finite = np.isfinite(seg)
    if finite.sum() < 2:
        return {
            "Mean": np.nan, "AUC": np.nan, "AUC_abs": np.nan, "RMS": np.nan, "SD": np.nan,
            "PeakSigned": np.nan, "PeakPos": np.nan, "PeakPosWinMean2s": np.nan, "PeakNeg": np.nan,
            "PeakAbs": np.nan, "PeakToPeak": np.nan, "PeakWinMean2s": np.nan
        }

    x = seg[finite].astype(float)

    mean_val = float(np.nanmean(x))
    pos_peak = float(np.nanmax(x))          # single positive max
    neg_peak = float(np.nanmin(x))          # single negative min
    abs_peak = float(np.nanmax(np.abs(x)))  # largest absolute
    auc      = float(np.trapz(x, dx=1.0 / sf))
    auc_abs  = float(np.trapz(np.abs(x), dx=1.0 / sf))
    rms      = float(np.sqrt(np.nanmean(x**2)))
    sd       = float(np.nanstd(x))
    ptp      = float(pos_peak - neg_peak)

    # --- NEW: windowed mean around POSITIVE peak (2 s total) ---
    i_pos = int(np.nanargmax(x))
    win_samples = max(1, int(round(2.0 * sf)))
    start = max(0, i_pos - win_samples // 2)
    end   = min(len(x), start + win_samples)
    win_pos = x[start:end]
    pos_peak_winmean = float(np.nanmean(win_pos)) if np.isfinite(win_pos).any() else np.nan

    # --- Signed peak using whichever has greater absolute magnitude ---
    signed_peak = pos_peak if abs(pos_peak) >= abs(neg_peak) else neg_peak

    # --- Existing "PeakWinMean2s" (around absolute max) ---
    if np.all(~np.isfinite(seg)):
        peak_win_mean_2s = np.nan
    else:
        i_center = int(np.nanargmax(np.abs(seg)))
        n = seg.size
        start = max(0, i_center - win_samples // 2)
        end = min(n, start + win_samples)
        w = seg[start:end]
        finite_in_win = np.isfinite(w)
        peak_win_mean_2s = float(np.nanmean(w)) if finite_in_win.sum() >= max(1, int(0.5 * win_samples)) else np.nan

    return {
        "Mean": mean_val, "AUC": auc, "AUC_abs": auc_abs, "RMS": rms, "SD": sd,
        "PeakSigned": signed_peak,
        "PeakPos": pos_peak,                 # original single-sample
        "PeakPosWinMean2s": pos_peak_winmean,  # new windowed version
        "PeakNeg": neg_peak,
        "PeakAbs": abs_peak, "PeakToPeak": ptp,
        "PeakWinMean2s": peak_win_mean_2s     # renamed from PeakWinMean1s
    }

def write_qc(subject: str,
             sci_df: pd.DataFrame,
             trial_qc_df: pd.DataFrame,
             baseline_info: dict,
             chanvar_qc_df: pd.DataFrame = None):
    Path(OUT_QC_DIR).mkdir(parents=True, exist_ok=True)
    if sci_df is not None and not sci_df.empty:
        sci_df.to_csv(Path(OUT_QC_DIR) / f"{subject}_sci.csv", index=False)
    if chanvar_qc_df is not None and not chanvar_qc_df.empty:
        chanvar_qc_df.to_csv(Path(OUT_QC_DIR) / f"{subject}_chanvar_qc.csv", index=False)
    if baseline_info:
        pd.DataFrame(baseline_info, index=[0]).to_csv(Path(OUT_QC_DIR) / f"{subject}_baseline.csv", index=False)
    if trial_qc_df is not None and not trial_qc_df.empty:
        trial_qc_df.to_csv(Path(OUT_QC_DIR) / f"{subject}_trial_qc.csv", index=False)

# --------- NEW: pair pruning etc. (unchanged) ---------

def apply_linear_detrend_od(od: mne.io.BaseRaw) -> mne.io.BaseRaw:
    n_times = od.n_times
    t_idx = np.arange(n_times)
    def _detrend_vec(x):
        xx = np.asarray(x, dtype=float)
        finite = np.isfinite(xx)
        if finite.sum() < 2:
            return xx
        p = np.polyfit(t_idx[finite], xx[finite], 1)
        trend = np.polyval(p, t_idx)
        return xx - trend
    picks = pick_fnirs_channels(od)
    od.apply_function(_detrend_vec, picks=picks, channel_wise=True, n_jobs=1, verbose=False)
    return od

def postfilter_channel_qc_drop(od: mne.io.BaseRaw,
                               metric: str = QC_METRIC,
                               mult: float = QC_MULT,
                               min_range: float = QC_MIN_RANGE):
    picks = pick_fnirs_channels(od)
    data = od.get_data(picks=picks)
    chs  = np.array(od.ch_names)[picks]
    vals = []
    for i in range(data.shape[0]):
        x = data[i]
        finite = np.isfinite(x)
        if finite.sum() < 2:
            vals.append(np.nan); continue
        if metric.lower().startswith("var"):
            v = float(np.nanvar(x[finite]))
        else:
            v = float(np.nanmax(x[finite]) - np.nanmin(x[finite]))
        if v < min_range:
            v = min_range
        vals.append(v)
    vals = np.asarray(vals, float)
    med  = np.nanmedian(vals)
    thresh = mult * med if np.isfinite(med) and med > 0 else np.inf
    drop_mask = vals > thresh
    drop_names = list(chs[drop_mask])
    qc_df = pd.DataFrame({
        "Channel": chs,
        "QC_Metric": metric,
        "Value": vals,
        "MedianAcrossCh": med,
        "Threshold": thresh,
        "Drop": drop_mask
    })
    if drop_names:
        od.drop_channels(drop_names)
    return drop_names, qc_df

def prune_and_order_od_for_mbl(od: mne.io.BaseRaw):
    picks = mne.pick_types(od.info, fnirs=True, stim=False, eeg=False, meg=False)
    pair_map = {}
    for idx in picks:
        name = od.ch_names[idx]
        pair = _pair_label(name)
        freq = float(od.info["chs"][idx]["loc"][9])  # wavelength freq
        pair_map.setdefault(pair, []).append((idx, name, freq))
    keep_names = []
    dropped_pairs = set()
    for pair, lst in pair_map.items():
        by_f = {}
        for idx, nm, f in lst:
            by_f.setdefault(round(f), []).append((idx, nm, f))
        if len(by_f) < 2:
            dropped_pairs.add(pair)
            continue
        chosen = []
        for fkey in sorted(by_f.keys())[:2]:
            chosen.append(sorted(by_f[fkey], key=lambda x: x[1])[0])
        chosen = sorted(chosen, key=lambda x: x[2])  # order by freq
        keep_names.extend([nm for _, nm, _ in chosen])
        for fkey, arr in by_f.items():
            if len(arr) > 1:
                dropped_pairs.add(pair)
    if not keep_names:
        return od, dropped_pairs
    od.pick_channels(keep_names)
    pair_freq_list = []
    for nm in od.ch_names:
        ch_idx = od.ch_names.index(nm)
        pair = _pair_label(nm)
        freq = float(od.info["chs"][ch_idx]["loc"][9])
        pair_freq_list.append((pair, freq, nm))
    pair_freq_list.sort(key=lambda t: (t[0], t[1]))
    od.reorder_channels([nm for _, _, nm in pair_freq_list])
    if dropped_pairs:
        ex = ", ".join(sorted(list(dropped_pairs))[:6])
        print(f"[PAIR] Pruned/standardized wavelengths for {len(dropped_pairs)} pairs (e.g., {ex}{'...' if len(dropped_pairs)>6 else ''})")
    return od, dropped_pairs

# ------------- PROCESSING ---------------

def preprocess_to_hb(raw: mne.io.BaseRaw, file_name: str):
    od = mne.preprocessing.nirs.optical_density(raw)

    all_pairs = {_pair_label(ch) for ch in od.ch_names}
    _, short_names = separate_long_short(od)
    short_pairs = {_pair_label(ch) for ch in short_names}

    if DO_OD_LINEAR_DETREND:
        try:
            od = apply_linear_detrend_od(od)
        except Exception as e:
            print(f"[Detrend] Skipped ({e})")

    sci_df = pd.DataFrame()
    sci_bad_pairs = set()
    if DO_SCI:
        try:
            from mne.preprocessing.nirs import scalp_coupling_index
            sci_vals = scalp_coupling_index(od)
            sci_df = pd.DataFrame({"Channel": od.ch_names, "SCI": sci_vals})
            bad_od_chs = [ch for ch, v in zip(od.ch_names, sci_vals) if np.isfinite(v) and v < SCI_THRESHOLD]
            sci_bad_pairs = {_pair_label(ch) for ch in bad_od_chs}
            if bad_od_chs:
                print(f"[SCI] Dropping {len(bad_od_chs)} OD chans (SCI < {SCI_THRESHOLD})")
                od.info["bads"] = list(set(od.info.get("bads", [])) | set(bad_od_chs))
                od.drop_channels(bad_od_chs)
        except Exception as e:
            print(f"[SCI] Skipped (reason: {e})")
            sci_df = pd.DataFrame({"Channel": od.ch_names, "SCI": np.nan})

    if DO_TDDR:
        try:
            od = mne.preprocessing.nirs.temporal_derivative_distribution_repair(od)
        except Exception as e:
            print(f"[TDDR] Skipped ({e})")

    if DO_SCR:
        _, short_names_now = separate_long_short(od)
        if len(short_names_now) >= 1:
            try:
                od = mne_nirs.signal_enhancement.short_channel_regression(od)
            except Exception as e:
                print(f"[SCR] Skipped (reason: {e})")
        else:
            print("[SCR] No short channels available (or removed).")

    if DO_BPF:
        try:
            od = od.filter(l_freq=BPF_LF, h_freq=BPF_HF, picks="all",
                           method='iir', iir_params=dict(order=IIR_ORDER, ftype='butter'))
        except Exception as e:
            print(f"[BPF] Skipped ({e})")

    chanvar_qc_df = pd.DataFrame()
    qcvar_bad_pairs = set()
    if DO_POSTFILTER_VAR_QC:
        try:
            dropped, chanvar_qc_df = postfilter_channel_qc_drop(od, metric=QC_METRIC, mult=QC_MULT, min_range=QC_MIN_RANGE)
            if dropped:
                qcvar_bad_pairs = {_pair_label(ch) for ch in dropped}
                print(f"[QCVAR] Dropped {len(dropped)} OD chans ({QC_METRIC} > {QC_MULT}× median)")
        except Exception as e:
            print(f"[QCVAR] Skipped ({e})")

    bad_power_spans = {}
    if DO_POWER_ANNOT:
        try:
            bad_power_spans = annotate_high_power_segments(od, win_sec=POWER_WIN_S,
                                                           step_frac=POWER_STEP_F, z_th=POWER_Z_TH)
        except Exception as e:
            print(f"[POWER] Annotation skipped ({e})")

    if KEEP_LONG_ONLY:
        long_names_now, _ = separate_long_short(od)
        if long_names_now:
            od.pick_channels([ch for ch in long_names_now if ch in od.ch_names])
        else:
            print("[WARN] No long channels found after processing.")

    pruned_pairs = set()
    try:
        od, pruned_pairs = prune_and_order_od_for_mbl(od)
    except Exception as e:
        print(f"[PAIR] Pair pruning skipped ({e})")

    hb = mne.preprocessing.nirs.beer_lambert_law(od)

    hbo_names = [ch for ch in hb.ch_names if str(ch).endswith(" hbo")]
    kept_pairs_final = {_pair_label(ch) for ch in hbo_names}

    pipe_meta = dict(
        all_pairs=all_pairs,
        short_pairs=short_pairs,
        sci_bad_pairs=sci_bad_pairs,
        qcvar_bad_pairs=qcvar_bad_pairs,
        pair_pruned_bad_pairs=pruned_pairs,
        kept_pairs_final=kept_pairs_final,
    )
    return hb, sci_df, bad_power_spans, pipe_meta, chanvar_qc_df

def find_single_baseline_span(raw: mne.io.BaseRaw):
    sf = raw.info['sfreq']
    candidates = []
    for on, dur, desc in zip(raw.annotations.onset, raw.annotations.duration, raw.annotations.description):
        if "BASELINE" in str(desc).upper():
            s = int((on - raw.first_time) * sf)
            e = int((on - raw.first_time + dur) * sf)
            s = max(0, s); e = min(raw.n_times, e)
            if e > s:
                candidates.append((s, e))
    if not candidates:
        return None
    return max(candidates, key=lambda t: t[1]-t[0])

def middle_window_of_span(s: int, e: int, desired_len_samp: int):
    L = e - s
    if desired_len_samp >= L:
        return s, e
    offset = (L - desired_len_samp) // 2
    ms = s + offset
    me = ms + desired_len_samp
    return ms, me

def process_subject_epochwise(subj: str, rows_for_subject: pd.DataFrame):
    fif_path = fif_path_for_subject(subj)
    if not fif_path.exists():
        print(f"[MISS] {subj}: {fif_path.name} not found → NaNs.")
        return (pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame(),
                {}, pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame())

    print(f"[LOAD] {subj}: {fif_path.name}")
    raw = mne.io.read_raw_fif(fif_path, preload=True, verbose=False)
    hb, sci_df, bad_power_spans, pipe_meta, chanvar_qc_df = preprocess_to_hb(raw, fif_path.name)

    baseline_info = {}
    base_mean = None
    if DO_BASELINE_SUBTRACT:
        base_span_samples = find_single_baseline_span(hb)
        if base_span_samples is None:
            print("[WARN] No 'Baseline' annotation found after processing; proceeding without baseline subtraction.")
            baseline_info = {"baseline_found": False}
        else:
            s_b, e_b = base_span_samples
            sf = hb.info['sfreq']
            desired_len_samp = int(BASELINE_MIDDLE_MIN * 60.0 * sf)
            ms, me = middle_window_of_span(s_b, e_b, desired_len_samp)
            picks = pick_fnirs_channels(hb)
            B = hb.get_data(picks=picks, start=ms, stop=me)
            base_mean = np.nanmean(B, axis=1, keepdims=True)
            baseline_info = {
                "baseline_found": True,
                "baseline_total_sec": (e_b - s_b) / sf,
                "baseline_middle_sec": (me - ms) / sf,
                "baseline_middle_start_sec": ms / sf + hb.first_time,
                "baseline_middle_end_sec": me / sf + hb.first_time
            }

    spans = parse_trials(hb)
    picks = pick_fnirs_channels(hb)
    ch_names = np.array(hb.ch_names)[picks]
    sf = hb.info['sfreq']
    hbo_idx = [i for i, ch in enumerate(ch_names) if str(ch).endswith(" hbo")]
    hbo_names = [ch_names[i] for i in hbo_idx]
    name_to_row = {name: i for i, name in enumerate(ch_names)}

    union_pairs = set(pipe_meta.get("kept_pairs_final", set())) \
                  | set(pipe_meta.get("sci_bad_pairs", set())) \
                  | set(pipe_meta.get("short_pairs", set())) \
                  | set(pipe_meta.get("qcvar_bad_pairs", set())) \
                  | set(pipe_meta.get("pair_pruned_bad_pairs", set()))

    summary_rows = []
    timeser_rows = []
    trial_qc = []
    channel_inclusion_rows = []

    # NEW accumulators for ROI outputs
    roi_summary_rows = []   # features on ROI-averaged waveform (Method 2)
    roi_timeser_rows = []   # ROI-averaged waveform itself (Method 2)
    roi_dual_rows = []      # Combined row per ROI: Method1 (ChWise_*) + Method2 (ROIWise_*)

    for _, r in rows_for_subject.iterrows():
        act = r["Activity"].upper()
        stim = r["Stimulation"].upper()
        tr   = r["Trial"].upper()
        key = (act, stim, tr)

        # If missing trial annotation: record NaNs for channel features and mark all ROIs missing
        if key not in spans:
            for pair in sorted(union_pairs):
                channel_inclusion_rows.append({
                    "trialID": r["trialID"], "Subject": r["Subject"],
                    "Activity": r["Activity"], "Stimulation": r["Stimulation"], "Trial": r["Trial"],
                    "Channel": f"{pair} hbo", "Kept": False, "Reason": "missing_annotation"
                })
            for ch in hbo_names:
                summary_rows.append({**r.to_dict(), "Channel": ch,
                                     "Mean": np.nan, "PeakSigned": np.nan,
                                     "PeakPos": np.nan, "PeakNeg": np.nan, "PeakAbs": np.nan})
            trial_qc.append({"trialID": r["trialID"], "desc": None, "kept": False,
                             "reason": "missing_annotation"})
            continue

        s_samp, e_samp, desc = spans[key]

        # BAD overlap → skip trial (like original handling)
        if is_bad_overlap(hb, s_samp, e_samp):
            for pair in sorted(union_pairs):
                channel_inclusion_rows.append({
                    "trialID": r["trialID"], "Subject": r["Subject"],
                    "Activity": r["Activity"], "Stimulation": r["Stimulation"], "Trial": r["Trial"],
                    "Channel": f"{pair} hbo", "Kept": False, "Reason": "BAD_overlap"
                })
            for ch in hbo_names:
                summary_rows.append({**r.to_dict(), "Channel": ch,
                                     "Mean": np.nan, "PeakSigned": np.nan,
                                     "PeakPos": np.nan, "PeakNeg": np.nan, "PeakAbs": np.nan})
            trial_qc.append({"trialID": r["trialID"], "desc": desc, "kept": False,
                             "reason": "BAD_overlap"})
            continue

        # Extract trial segment matrix (n_channels x n_times), apply baseline
        X = hb.get_data(picks=picks, start=s_samp, stop=e_samp)
        if DO_BASELINE_SUBTRACT and base_mean is not None:
            X = X - base_mean

        # Power masking (if used): convert to NaNs in the HbO rows
        if DO_POWER_ANNOT and bad_power_spans:
            for od_ch, spans_list in bad_power_spans.items():
                pair = _pair_label(od_ch)
                hbo_name = f"{pair} hbo"
                row = name_to_row.get(hbo_name, None)
                if row is None:
                    continue
                for s_bad, e_bad in spans_list:
                    s_overlap = max(s_bad, s_samp)
                    e_overlap = min(e_bad, e_samp)
                    if e_overlap > s_overlap:
                        i0 = s_overlap - s_samp
                        i1 = e_overlap - s_samp
                        X[row, i0:i1] = np.nan

        # Per-pair inclusion (as before)
        for pair in sorted(union_pairs):
            hbo_name = f"{pair} hbo"
            if pair in pipe_meta.get("short_pairs", set()):
                channel_inclusion_rows.append({**r.to_dict(),
                    "Channel": hbo_name, "Kept": False, "Reason": "short"}); continue
            if pair in pipe_meta.get("sci_bad_pairs", set()):
                channel_inclusion_rows.append({**r.to_dict(),
                    "Channel": hbo_name, "Kept": False, "Reason": "SCI"}); continue
            if pair in pipe_meta.get("qcvar_bad_pairs", set()):
                channel_inclusion_rows.append({**r.to_dict(),
                    "Channel": hbo_name, "Kept": False, "Reason": "qcvar"}); continue
            if pair in pipe_meta.get("pair_pruned_bad_pairs", set()):
                channel_inclusion_rows.append({**r.to_dict(),
                    "Channel": hbo_name, "Kept": False, "Reason": "unpaired_wavelength"}); continue
            row_idx = name_to_row.get(hbo_name, None)
            if row_idx is None:
                channel_inclusion_rows.append({**r.to_dict(),
                    "Channel": hbo_name, "Kept": False, "Reason": "not_present_after_pipeline"})
                continue
            seg_tmp = X[row_idx]
            if seg_tmp.size == 0:
                channel_inclusion_rows.append({**r.to_dict(),
                    "Channel": hbo_name, "Kept": False, "Reason": "empty_segment"})
                continue
            frac_nan = float(np.mean(~np.isfinite(seg_tmp)))
            if frac_nan >= POWER_MASK_DROP_FRAC:
                channel_inclusion_rows.append({**r.to_dict(),
                    "Channel": hbo_name, "Kept": False, "Reason": "power_masked_majority"})
            else:
                channel_inclusion_rows.append({**r.to_dict(),
                    "Channel": hbo_name, "Kept": True, "Reason": "ok"})

        # ---- Per-channel features & time series ----
        ch_feats_for_trial = {}   # cache for Method 1 aggregation
        for ch in hbo_names:
            row_idx = name_to_row[ch]
            seg = X[row_idx]
            feats = compute_epoch_features(seg, sf)
            summary_rows.append({**r.to_dict(), "Channel": ch, **feats})
            ch_feats_for_trial[ch] = feats  # cache
            if EXPORT_TIMESER:
                timeser_rows.append({**r.to_dict(), "Channel": ch, "sfreq": float(sf),
                                     "waveform": seg.astype(float)})

        # ---- Build set of kept HbO channels for THIS trial ----
        kept_this_trial = {
            d["Channel"] for d in channel_inclusion_rows
            if (d["trialID"] == r["trialID"]) and d.get("Kept", False) and str(d["Channel"]).endswith(" hbo")
        }

        # ---- ROI Method 2 (ROI-wise waveform -> features) ----
        roi_method2_map = {}  # ROI -> (feats dict, N_ch)
        for roi_name, roi_chlist in brain_regions.items():
            # Find all contributing channel indices for this ROI
            idxs = []
            for ch_name in roi_chlist:
                if ch_name in kept_this_trial:
                    row_idx = name_to_row.get(ch_name, None)
                    if row_idx is not None:
                        idxs.append(row_idx)

            N_ch = len(idxs)
            if N_ch == 0:
                continue  # nothing kept for this ROI in this trial

            # ROI waveform = mean across contributing channels (row-wise mean)
            roi_seg = np.nanmean(X[np.array(idxs), :], axis=0)
            feats_roi = compute_epoch_features(roi_seg, sf)

            roi_summary_rows.append({
                **r.to_dict(),
                "ROI": roi_name,
                "N_ch": int(N_ch),   # how many channels contributed
                **feats_roi
            })
            roi_method2_map[roi_name] = (feats_roi, int(N_ch))

            if EXPORT_TIMESER:
                roi_timeser_rows.append({
                    **r.to_dict(),
                    "ROI": roi_name,
                    "sfreq": float(sf),
                    "waveform": roi_seg.astype(float),
                    "N_ch": int(N_ch)
                })

        # ---- ROI Method 1 (channel-wise features -> average across ROI) and combine with Method 2 ----
        feature_keys = ["Mean","AUC","AUC_abs","RMS","SD","PeakSigned","PeakPos","PeakPosWinMean2s","PeakNeg","PeakAbs","PeakToPeak","PeakWinMean2s"]



        for roi_name, roi_chlist in brain_regions.items():
            # channels in ROI that are both present & Kept this trial and we have feats for them
            ch_in_roi_kept = [ch for ch in roi_chlist if ch in kept_this_trial and ch in ch_feats_for_trial]
            n_ch_m1 = len(ch_in_roi_kept)

            # Method 1 aggregation: mean across channel features (NaN-robust)
            chwise_agg = {}
            if n_ch_m1 > 0:
                for fk in feature_keys:
                    vals = [ch_feats_for_trial[ch].get(fk, np.nan) for ch in ch_in_roi_kept]
                    chwise_agg[f"ChWise_{fk}"] = float(np.nanmean(vals)) if np.isfinite(vals).any() else np.nan
            else:
                for fk in feature_keys:
                    chwise_agg[f"ChWise_{fk}"] = np.nan
            chwise_agg["ChWise_N_ch"] = int(n_ch_m1)

            # Method 2 values (if available for this ROI)
            roiwise_agg = {}
            if roi_name in roi_method2_map:
                feats_roi, n_ch_m2 = roi_method2_map[roi_name]
                for fk in feature_keys:
                    roiwise_agg[f"ROIWise_{fk}"] = feats_roi.get(fk, np.nan)
                roiwise_agg["ROIWise_N_ch"] = int(n_ch_m2)
            else:
                for fk in feature_keys:
                    roiwise_agg[f"ROIWise_{fk}"] = np.nan
                roiwise_agg["ROIWise_N_ch"] = np.nan

            roi_dual_rows.append({
                **r.to_dict(),
                "ROI": roi_name,
                **chwise_agg,
                **roiwise_agg
            })

        trial_qc.append({"trialID": r["trialID"], "desc": desc, "kept": True, "reason": "ok"})

    return (pd.DataFrame(summary_rows),
            pd.DataFrame(timeser_rows),
            sci_df,
            pd.DataFrame(trial_qc),
            baseline_info,
            pd.DataFrame(channel_inclusion_rows),
            chanvar_qc_df,
            pd.DataFrame(roi_summary_rows),
            pd.DataFrame(roi_timeser_rows),
            pd.DataFrame(roi_dual_rows))

# -------- Visualization (per-channel; unchanged) --------

def _ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def _moving_average(x: np.ndarray, k: int) -> np.ndarray:
    if k <= 1:
        return x
    return np.convolve(x, np.ones(k, dtype=float)/k, mode='same')

def plot_hbo_four_trials_windows(ts_df_unexpanded: pd.DataFrame,
                                 subjects: list,
                                 activities: list,
                                 smooth_sec: float = 0.0,
                                 time_units: str = "s",
                                 save_figs: bool = True,
                                 show_windows: bool = False):
    if ts_df_unexpanded.empty:
        print("[PLOT] No timeseries data to plot.")
        return
    _ensure_dir(Path(OUT_FIG_DIR))
    for subj in subjects:
        for act in activities:
            fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
            axes = axes.ravel()
            title = f"{subj} · {act} · HbO (final, baseline-corrected)"
            fig.suptitle(title)
            for pi, (stim, tr) in enumerate([(STIMS_ORDER[0], TRIALS_ORDER[0]),
                                             (STIMS_ORDER[1], TRIALS_ORDER[0]),
                                             (STIMS_ORDER[0], TRIALS_ORDER[1]),
                                             (STIMS_ORDER[1], TRIALS_ORDER[1])]):
                ax = axes[pi]
                df = ts_df_unexpanded[
                    (ts_df_unexpanded["Subject"] == subj) &
                    (ts_df_unexpanded["Activity"].str.upper() == act.upper()) &
                    (ts_df_unexpanded["Stimulation"].str.upper() == stim.upper()) &
                    (ts_df_unexpanded["Trial"].str.upper() == tr.upper())
                ].copy()
                if not df.empty:
                    df = df[df["Channel"].astype(str).str.endswith(" hbo")]
                if df.empty:
                    ax.set_title(f"{stim}-{tr}: (missing)")
                    ax.set_xlabel("Time (s)" if time_units.lower().startswith("s") else "Samples")
                    ax.set_ylabel("HbO (Δ units)")
                    ax.grid(alpha=0.2); continue
                lengths = [len(w) for w in df["waveform"]]
                L = int(min(lengths))
                sf = float(df["sfreq"].iloc[0]) if "sfreq" in df.columns else 1.0
                if time_units.lower().startswith("s"):
                    t = np.arange(L) / sf
                    ax.set_xlabel("Time (s)")
                else:
                    t = np.arange(L)
                    ax.set_xlabel("Samples")
                k = max(1, int(round(smooth_sec * sf))) if (smooth_sec and smooth_sec > 0) else 1
                for _, row in df.iterrows():
                    x = np.asarray(row["waveform"], dtype=float)[:L]
                    if k > 1:
                        x = _moving_average(x, k)
                    ax.plot(t, x, linewidth=1.0, alpha=0.9)
                ax.set_title(f"{stim}-{tr} (Nch={len(df)})")
                ax.set_ylabel("HbO (Δ units)")
                ax.grid(alpha=0.2)
            fig.tight_layout(rect=[0, 0.03, 1, 0.95])
            if save_figs:
                out_png = Path(OUT_FIG_DIR) / f"{subj}_{act}_HbO_four_trials.png"
                fig.savefig(out_png, dpi=150)
                print(f"[PLOT] Saved {out_png}")
            if show_windows:
                plt.show()
            else:
                plt.close(fig)

# -------- NEW: ROI overlay plots (10 ROIs per panel) --------

def plot_roi_four_trials_windows(roi_ts_df_unexpanded: pd.DataFrame,
                                 subjects: list,
                                 activities: list,
                                 smooth_sec: float = 0.0,
                                 time_units: str = "s",
                                 save_figs: bool = True,
                                 show_windows: bool = False):
    """
    For each (Subject, Activity), create a 2x2 panel:
      [NG-T1, G-T1; NG-T2, G-T2]
    Each panel overlays one line per ROI (≈10 lines), with a single legend.
    """
    if roi_ts_df_unexpanded.empty:
        print("[ROI-PLOT] No ROI timeseries to plot.")
        return

    out_dir = Path(OUT_ROI_FIG_DIR)
    _ensure_dir(out_dir)

    # Fixed ROI order for color consistency
    roi_order = list(brain_regions.keys())

    for subj in subjects:
        for act in activities:
            fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
            axes = axes.ravel()
            fig.suptitle(f"{subj} · {act} · HbO ROI overlays (baseline-corrected)")

            # Keep global handles/labels for shared legend
            handles_labels = None

            for pi, (stim, tr) in enumerate([(STIMS_ORDER[0], TRIALS_ORDER[0]),
                                             (STIMS_ORDER[1], TRIALS_ORDER[0]),
                                             (STIMS_ORDER[0], TRIALS_ORDER[1]),
                                             (STIMS_ORDER[1], TRIALS_ORDER[1])]):
                ax = axes[pi]
                df = roi_ts_df_unexpanded[
                    (roi_ts_df_unexpanded["Subject"] == subj) &
                    (roi_ts_df_unexpanded["Activity"].str.upper() == act.upper()) &
                    (roi_ts_df_unexpanded["Stimulation"].str.upper() == stim.upper()) &
                    (roi_ts_df_unexpanded["Trial"].str.upper() == tr.upper())
                ].copy()

                if df.empty:
                    ax.set_title(f"{stim}-{tr}: (missing)")
                    ax.set_xlabel("Time (s)" if time_units.lower().startswith("s") else "Samples")
                    ax.set_ylabel("HbO (Δ units)")
                    ax.grid(alpha=0.2); continue

                # shortest length across ROIs in this panel
                lengths = [len(w) for w in df["waveform"]]
                L = int(min(lengths))
                sf = float(df["sfreq"].iloc[0]) if "sfreq" in df.columns else 1.0
                t = (np.arange(L) / sf) if time_units.lower().startswith("s") else np.arange(L)
                ax.set_xlabel("Time (s)" if time_units.lower().startswith("s") else "Samples")
                ax.set_ylabel("HbO (Δ units)")

                k = max(1, int(round(smooth_sec * sf))) if (smooth_sec and smooth_sec > 0) else 1

                # Plot ROIs in consistent order, skipping missing ones
                for roi in roi_order:
                    dfr = df[df["ROI"] == roi]
                    if dfr.empty:
                        continue
                    x = np.asarray(dfr["waveform"].iloc[0], dtype=float)[:L]
                    if k > 1:
                        x = _moving_average(x, k)
                    line, = ax.plot(t, x, linewidth=1.4, alpha=0.95, label=roi)
                    if handles_labels is None:
                        handles_labels = ([line], [roi])
                    else:
                        handles_labels[0].append(line)
                        handles_labels[1].append(roi)

                nroi = df["ROI"].nunique()
                ax.set_title(f"{stim}-{tr} (NROI={nroi})")
                ax.grid(alpha=0.25)

            # Shared legend outside figure (right)
            if handles_labels and len(handles_labels[0]) > 0:
                # Deduplicate while preserving order (one entry per ROI)
                seen = set()
                uniq_handles, uniq_labels = [], []
                for h, lb in zip(handles_labels[0], handles_labels[1]):
                    if lb in seen:
                        continue
                    seen.add(lb)
                    uniq_handles.append(h)
                    uniq_labels.append(lb)
                fig.legend(uniq_handles, uniq_labels, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)

            fig.tight_layout(rect=[0, 0.03, 0.88, 0.95])  # leave space for side legend
            if save_figs:
                out_png = Path(OUT_ROI_FIG_DIR) / f"{subj}_{act}_HbO_four_trials_ROI.png"
                fig.savefig(out_png, dpi=150, bbox_inches="tight")
                print(f"[ROI-PLOT] Saved {out_png}")
            if show_windows:
                plt.show()
            else:
                plt.close(fig)

# --------------- MAIN -------------------

def main():
    # Load template and select rows
    template = pd.read_excel(TEMPLATE_XLSX)
    template = template[["trialID","Subject","Fatigue","Activity","Stimulation","Trial"]]

    selected = template[template["Subject"].isin(SUBJECTS_CF + SUBJECTS_SD)].copy()
    selected["SubjectNum"]   = selected["Subject"].str.extract(r"(\d+)").astype(int)
    selected["Activity"]     = selected["Activity"].astype(pd.CategoricalDtype(categories=ACTIVITIES, ordered=True))
    selected["Stimulation"]  = selected["Stimulation"].astype(pd.CategoricalDtype(categories=STIMS_ORDER, ordered=True))
    selected["Trial"]        = selected["Trial"].astype(pd.CategoricalDtype(categories=TRIALS_ORDER, ordered=True))
    selected.sort_values(["SubjectNum","Activity","Stimulation","Trial"], inplace=True)
    selected.drop(columns=["SubjectNum"], inplace=True)

    all_summary = []
    all_timeser = []
    all_roi_summary = []     # Method 2 (ROI-wise)
    all_roi_timeser = []     # Method 2 (ROI-wise)
    all_roi_dual = []        # Combined Method 1 vs Method 2

    for subj, group in selected.groupby("Subject", sort=False):
        (sdf, tsdf, sci_df, trial_qc_df, baseline_info,
         ch_incl_df, chanvar_qc_df, roi_sdf, roi_tsdf, roi_dual_df) = process_subject_epochwise(
            subj, group.reset_index(drop=True)
        )

        all_summary.append(sdf)
        all_timeser.append(tsdf)
        all_roi_summary.append(roi_sdf)
        all_roi_timeser.append(roi_tsdf)
        all_roi_dual.append(roi_dual_df)

        write_qc(subj, sci_df, trial_qc_df, baseline_info, chanvar_qc_df)
        qc_dir = Path(OUT_QC_DIR); qc_dir.mkdir(parents=True, exist_ok=True)
        if not ch_incl_df.empty:
            ch_incl_df.to_csv(qc_dir / f"{subj}_channel_inclusion_by_trial.csv", index=False)

    # ===== Per-channel SUMMARY export (unchanged) =====
    summary = pd.concat(all_summary, ignore_index=True) if all_summary else pd.DataFrame()
    if not summary.empty:
        summary["SubjectNum"]  = summary["Subject"].str.extract(r"(\d+)").astype(int)
        summary["Activity"]    = summary["Activity"].astype(pd.CategoricalDtype(categories=ACTIVITIES, ordered=True))
        summary["Stimulation"] = summary["Stimulation"].astype(pd.CategoricalDtype(categories=STIMS_ORDER, ordered=True))
        summary["Trial"]       = summary["Trial"].astype(pd.CategoricalDtype(categories=TRIALS_ORDER, ordered=True))
        summary.sort_values(["SubjectNum","Activity","Stimulation","Trial","Channel"], inplace=True)
        summary.drop(columns=["SubjectNum"], inplace=True)
        lead = ["trialID","Subject","Fatigue","Activity","Stimulation","Trial","Channel"]
        cols = lead + ["Mean","AUC","AUC_abs","RMS","SD","PeakSigned","PeakPos","PeakPosWinMean2s","PeakNeg","PeakAbs","PeakToPeak","PeakWinMean2s"]
        summary = summary[cols]
        summary.to_excel(OUT_SUMMARY, index=False)
        print(f"[SAVED] {OUT_SUMMARY} ({len(summary)} rows)")
    else:
        print("[WARN] Summary is empty.")

    # ===== Per-channel TIMESERIES export (unchanged) =====
    ts_unexpanded = pd.concat(all_timeser, ignore_index=True) if all_timeser else pd.DataFrame()
    if not ts_unexpanded.empty:
        temp_ts = ts_unexpanded.copy()
        temp_ts["SubjectNum"]  = temp_ts["Subject"].str.extract(r"(\d+)").astype(int)
        temp_ts["Activity"]    = temp_ts["Activity"].astype(pd.CategoricalDtype(categories=ACTIVITIES, ordered=True))
        temp_ts["Stimulation"] = temp_ts["Stimulation"].astype(pd.CategoricalDtype(categories=STIMS_ORDER, ordered=True))
        temp_ts["Trial"]       = temp_ts["Trial"].astype(pd.CategoricalDtype(categories=TRIALS_ORDER, ordered=True))
        temp_ts.sort_values(["SubjectNum","Activity","Stimulation","Trial","Channel"], inplace=True)
        temp_ts.drop(columns=["SubjectNum"], inplace=True)
        maxlen = max(len(wf) for wf in temp_ts["waveform"])
        ts_cols = [f"t{i}" for i in range(maxlen)]
        expanded = pd.DataFrame(temp_ts["waveform"].apply(lambda a: np.asarray(a, float)).to_list(), columns=ts_cols)
        temp_ts = pd.concat([temp_ts.drop(columns=["waveform"]).reset_index(drop=True),
                             expanded.reset_index(drop=True)], axis=1)
        lead = ["trialID","Subject","Fatigue","Activity","Stimulation","Trial","Channel"]
        temp_ts = temp_ts[lead + ts_cols]
        try:
            temp_ts.to_excel(OUT_TIMESER, index=False)
            print(f"[SAVED] {OUT_TIMESER} ({len(temp_ts)} rows)")
        except Exception as e:
            fb = OUT_TIMESER.rsplit(".", 1)[0] + ".csv"
            temp_ts.to_csv(fb, index=False)
            print(f"[SAVED] {fb} (Excel engine unavailable; reason: {e})")
    else:
        print("[WARN] No time series rows (empty all_timeser).")

    # ===== NEW: ROI Avg->Feat SUMMARY export (with N_ch) [Method 2] =====
    roi_summary = pd.concat(all_roi_summary, ignore_index=True) if all_roi_summary else pd.DataFrame()
    if not roi_summary.empty:
        roi_summary["SubjectNum"]  = roi_summary["Subject"].str.extract(r"(\d+)").astype(int)
        roi_summary["Activity"]    = roi_summary["Activity"].astype(pd.CategoricalDtype(categories=ACTIVITIES, ordered=True))
        roi_summary["Stimulation"] = roi_summary["Stimulation"].astype(pd.CategoricalDtype(categories=STIMS_ORDER, ordered=True))
        roi_summary["Trial"]       = roi_summary["Trial"].astype(pd.CategoricalDtype(categories=TRIALS_ORDER, ordered=True))
        roi_summary.sort_values(["SubjectNum","Activity","Stimulation","Trial","ROI"], inplace=True)
        roi_summary.drop(columns=["SubjectNum"], inplace=True)
        lead_roi = ["trialID","Subject","Fatigue","Activity","Stimulation","Trial","ROI","N_ch"]
        feat_cols = ["Mean","AUC","AUC_abs","RMS","SD","PeakSigned","PeakPos","PeakPosWinMean2s","PeakNeg","PeakAbs","PeakToPeak","PeakWinMean2s"]
        roi_summary = roi_summary[lead_roi + feat_cols]
        roi_summary.to_excel(OUT_ROI_SUMMARY, index=False)
        print(f"[SAVED] {OUT_ROI_SUMMARY} ({len(roi_summary)} rows)")
    else:
        print("[WARN] ROI Avg->Feat summary is empty.")

    # ===== NEW: ROI TIMESERIES export (wide) [Method 2] =====
    roi_ts_unexpanded = pd.concat(all_roi_timeser, ignore_index=True) if all_roi_timeser else pd.DataFrame()
    if not roi_ts_unexpanded.empty:
        temp_ts = roi_ts_unexpanded.copy()
        temp_ts["SubjectNum"]  = temp_ts["Subject"].str.extract(r"(\d+)").astype(int)
        temp_ts["Activity"]    = temp_ts["Activity"].astype(pd.CategoricalDtype(categories=ACTIVITIES, ordered=True))
        temp_ts["Stimulation"] = temp_ts["Stimulation"].astype(pd.CategoricalDtype(categories=STIMS_ORDER, ordered=True))
        temp_ts["Trial"]       = temp_ts["Trial"].astype(pd.CategoricalDtype(categories=TRIALS_ORDER, ordered=True))
        temp_ts.sort_values(["SubjectNum","Activity","Stimulation","Trial","ROI"], inplace=True)
        temp_ts.drop(columns=["SubjectNum"], inplace=True)
        maxlen = max(len(wf) for wf in temp_ts["waveform"])
        ts_cols = [f"t{i}" for i in range(maxlen)]
        expanded = pd.DataFrame(temp_ts["waveform"].apply(lambda a: np.asarray(a, float)).to_list(), columns=ts_cols)
        temp_ts = pd.concat([temp_ts.drop(columns=["waveform"]).reset_index(drop=True),
                             expanded.reset_index(drop=True)], axis=1)
        lead = ["trialID","Subject","Fatigue","Activity","Stimulation","Trial","ROI","N_ch"]
        temp_ts = temp_ts[lead + ts_cols]
        try:
            temp_ts.to_excel(OUT_ROI_TIMESER, index=False)
            print(f"[SAVED] {OUT_ROI_TIMESER} ({len(temp_ts)} rows)")
        except Exception as e:
            fb = OUT_ROI_TIMESER.rsplit(".", 1)[0] + ".csv"
            temp_ts.to_csv(fb, index=False)
            print(f"[SAVED] {fb} (Excel engine unavailable; reason: {e})")
    else:
        print("[WARN] No ROI time series rows.")

    # ===== NEW: Combined per-ROI features (Method 1 vs Method 2) =====
    roi_dual_all = pd.concat(all_roi_dual, ignore_index=True) if all_roi_dual else pd.DataFrame()
    if not roi_dual_all.empty:
        roi_dual_all["SubjectNum"]  = roi_dual_all["Subject"].str.extract(r"(\d+)").astype(int)
        roi_dual_all["Activity"]    = roi_dual_all["Activity"].astype(pd.CategoricalDtype(categories=ACTIVITIES, ordered=True))
        roi_dual_all["Stimulation"] = roi_dual_all["Stimulation"].astype(pd.CategoricalDtype(categories=STIMS_ORDER, ordered=True))
        roi_dual_all["Trial"]       = roi_dual_all["Trial"].astype(pd.CategoricalDtype(categories=TRIALS_ORDER, ordered=True))
        roi_dual_all.sort_values(["SubjectNum","Activity","Stimulation","Trial","ROI"], inplace=True)
        roi_dual_all.drop(columns=["SubjectNum"], inplace=True)

        meta = ["trialID","Subject","Fatigue","Activity","Stimulation","Trial","ROI"]
        counts = ["ChWise_N_ch","ROIWise_N_ch"]
        chwise_cols = sorted([c for c in roi_dual_all.columns if c.startswith("ChWise_") and c not in counts])
        roiwise_cols = sorted([c for c in roi_dual_all.columns if c.startswith("ROIWise_") and c not in counts])

        roi_dual_all = roi_dual_all[meta + counts + chwise_cols + roiwise_cols]
        roi_dual_all.to_excel(OUT_ROI_METHODS_SUMMARY, index=False)
        print(f"[SAVED] {OUT_ROI_METHODS_SUMMARY} ({len(roi_dual_all)} rows)")
    else:
        print("[WARN] Combined ROI methods summary is empty.")

    # ===== Visualization =====
    if DO_PLOTS:
        # Existing per-channel plot
        plot_hbo_four_trials_windows(
            ts_df_unexpanded=ts_unexpanded,
            subjects=PLOT_SUBJECTS,
            activities=PLOT_ACTIVITIES,
            smooth_sec=PLOT_SMOOTH_SEC,
            time_units=PLOT_TIME_UNITS,
            save_figs=PLOT_SAVE_FIGS,
            show_windows=PLOT_SHOW
        )
        # NEW ROI overlay plot
        plot_roi_four_trials_windows(
            roi_ts_df_unexpanded=roi_ts_unexpanded,
            subjects=PLOT_SUBJECTS,
            activities=PLOT_ACTIVITIES,
            smooth_sec=PLOT_SMOOTH_SEC,
            time_units=PLOT_TIME_UNITS,
            save_figs=PLOT_SAVE_FIGS,
            show_windows=PLOT_SHOW
        )

    print("[DONE]")

if __name__ == "__main__":
    main()
